# M01 · ¿Roma acepta la batalla campal?
**Santino Garbino · Agustin Quesada | ChatGPT | Pista: Reglas**

Ejecutar todas las celdas. La última permite ingresar el caso nuevo. Código y CSV incluidos: no necesita descargar nada ni llamar a un modelo de IA. Son 20 casos **sintéticos** de la cátedra; no pronósticos de batallas reales.

**Partición inalterable:** 1–14 ajuste; 15–20 prueba. Las etiquetas de prueba aparecieron en la lectura pública de la ficha; no se declara un ensayo ciego. La búsqueda de umbrales usa exclusivamente entrenamiento. El bot de Telegram y LinkedIn siguen pendientes.

## 1. Copia de los datos oficiales
El identificador, el conjunto y `resultado_real` no entran en la función predictora.

In [1]:
from pathlib import Path
BASE = Path("trabajo_M01")
(BASE / "datos").mkdir(parents=True, exist_ok=True)
DATOS_OFICIALES = 'caso,conjunto,Relación de fuerzas (infantería romana / cartaginesa),Caballería propia (% del ejército),Terreno favorable (índice 0 a 10; 10 = llano propio y conocido),resultado_real\n1,entrenamiento,1.33,14,3,0\n2,entrenamiento,1.24,23,0,0\n3,entrenamiento,1.05,33,7,1\n4,entrenamiento,0.99,14,6,0\n5,entrenamiento,1.02,9,6,0\n6,entrenamiento,0.62,16,4,0\n7,entrenamiento,1.77,13,2,0\n8,entrenamiento,1.63,35,9,1\n9,entrenamiento,1.51,32,10,1\n10,entrenamiento,0.87,35,7,1\n11,entrenamiento,0.79,17,8,1\n12,entrenamiento,1.19,30,0,0\n13,entrenamiento,0.88,22,2,0\n14,entrenamiento,1.92,28,9,1\n15,prueba,1.36,18,6,1\n16,prueba,0.82,27,4,0\n17,prueba,1.33,24,5,0\n18,prueba,1.81,30,2,1\n19,prueba,0.81,10,3,0\n20,prueba,0.99,28,6,1\n'
(BASE / "datos" / "M01.csv").write_bytes(DATOS_OFICIALES.replace("\n", "\r\n").encode("utf-8"))
print("Datos locales preparados: 20 casos oficiales.")

Datos locales preparados: 20 casos oficiales.


## 2. Motor de reglas y funciones de evaluación
Python estándar. Se requieren dos señales favorables. El costo 5:1 es un supuesto didáctico explícito, no un valor de la ficha.

In [2]:
"""M01: reglas de corroboración, ajuste SOLO con entrenamiento y evaluación reproducible.
Python 3.9+; biblioteca estándar; no requiere internet ni servicios de IA.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import itertools
import json
import math
import statistics
from pathlib import Path
from typing import Any

COSTO_FP = 5
COSTO_FN = 1
VARIABLES = ('relacion', 'caballeria', 'terreno')
COLUMNAS = (
    'Relación de fuerzas (infantería romana / cartaginesa)',
    'Caballería propia (% del ejército)',
    'Terreno favorable (índice 0 a 10; 10 = llano propio y conocido)',
)


def numero(valor: Any) -> float:
    """Admite coma decimal en ingreso manual; rechaza booleanos y no finitos."""
    if isinstance(valor, bool):
        raise ValueError('Ingrese un número, no un booleano.')
    try:
        x = float(str(valor).strip().replace(',', '.'))
    except (ValueError, TypeError) as exc:
        raise ValueError(f'Valor no numérico: {valor!r}') from exc
    if not math.isfinite(x):
        raise ValueError('No se admiten NaN ni infinitos.')
    return x


def validar_entrada(relacion: Any, caballeria: Any, terreno: Any) -> tuple[float, float, float]:
    r, c, t = map(numero, (relacion, caballeria, terreno))
    if r <= 0:
        raise ValueError('La relación de infantería debe ser mayor que cero.')
    if not 0 <= c <= 100:
        raise ValueError('La caballería debe expresarse como porcentaje entre 0 y 100.')
    if not 0 <= t <= 10:
        raise ValueError('El terreno debe estar entre 0 y 10.')
    return r, c, t


def cargar_datos(ruta: str | Path) -> list[dict]:
    with Path(ruta).open(encoding='utf-8-sig', newline='') as f:
        lector = csv.DictReader(f)
        esperadas = {'caso', 'conjunto', *COLUMNAS, 'resultado_real'}
        if set(lector.fieldnames or []) != esperadas:
            raise ValueError('El CSV no tiene las columnas oficiales de M01.')
        filas = []
        for fila in lector:
            valores = validar_entrada(*(fila[c] for c in COLUMNAS))
            y = int(fila['resultado_real'])
            if y not in (0, 1):
                raise ValueError('La etiqueta real debe ser 0 o 1.')
            filas.append(dict(caso=int(fila['caso']), conjunto=fila['conjunto'],
                              **dict(zip(VARIABLES, valores)), resultado_real=y))
    if len(filas) != 20 or {f['caso'] for f in filas} != set(range(1, 21)):
        raise ValueError('Se requieren exactamente los 20 casos oficiales, sin duplicados.')
    for f in filas:
        esperado = 'entrenamiento' if f['caso'] <= 14 else 'prueba'
        if f['conjunto'] != esperado:
            raise ValueError('No se permite cambiar la división 1–14 / 15–20.')
    return sorted(filas, key=lambda f: f['caso'])


def puntos_medios(valores: list[float]) -> list[float]:
    xs = sorted(set(valores))
    if len(xs) < 2:
        raise ValueError('Una variable constante no admite este ajuste.')
    return [round((a + b) / 2, 10) for a, b in zip(xs, xs[1:])]


def matriz(reales: list[int], predichos: list[int]) -> dict:
    if not reales or len(reales) != len(predichos):
        raise ValueError('Listas vacías o de distinta longitud.')
    if any(y not in (0, 1) for y in [*reales, *predichos]):
        raise ValueError('Las etiquetas deben ser binarias.')
    return {
        'VP': sum(y == 1 and p == 1 for y, p in zip(reales, predichos)),
        'FN': sum(y == 1 and p == 0 for y, p in zip(reales, predichos)),
        'FP': sum(y == 0 and p == 1 for y, p in zip(reales, predichos)),
        'VN': sum(y == 0 and p == 0 for y, p in zip(reales, predichos)),
    }


def metricas(reales: list[int], predichos: list[int]) -> dict:
    m = matriz(reales, predichos)
    vp, fn, fp, vn = (m[k] for k in ('VP', 'FN', 'FP', 'VN'))
    dividir = lambda a, b: a / b if b else None
    n = len(reales)
    return {**m, 'N': n,
            'exactitud': (vp + vn) / n,
            'precision': dividir(vp, vp + fp),
            'sensibilidad': dividir(vp, vp + fn),
            'especificidad': dividir(vn, vn + fp),
            'costo_total': COSTO_FP * fp + COSTO_FN * fn,
            'costo_medio': (COSTO_FP * fp + COSTO_FN * fn) / n}


def ajustar(entrenamiento: list[dict]) -> dict:
    """Busca umbrales sin recibir prueba. No se altera la familia por su resultado."""
    if len(entrenamiento) != 14 or {f['caso'] for f in entrenamiento} != set(range(1, 15)):
        raise ValueError('El ajuste exige SOLO los casos 1 a 14.')
    if any(f['conjunto'] != 'entrenamiento' for f in entrenamiento):
        raise ValueError('Está prohibido usar prueba en el ajuste.')
    X = [tuple(f[v] for v in VARIABLES) for f in entrenamiento]
    y = [f['resultado_real'] for f in entrenamiento]
    columnas = list(zip(*X))
    rangos = [max(xs) - min(xs) for xs in columnas]
    centros = [(max(xs) + min(xs)) / 2 for xs in columnas]
    candidatos = [puntos_medios(list(xs)) for xs in columnas]
    mejor, mejor_key, total = None, None, 0
    for umbrales in itertools.product(*candidatos):
        total += 1
        pred = [int(sum(a >= b for a, b in zip(x, umbrales)) >= 2) for x in X]
        m = metricas(y, pred)
        margenes = [(2 * real - 1) * statistics.median(
            (a - b) / rango for a, b, rango in zip(x, umbrales, rangos))
            for x, real in zip(X, y)]
        margen = min(margenes)
        distancia = sum(((u - centro) / rango) ** 2
                        for u, centro, rango in zip(umbrales, centros, rangos))
        # Redondeo SOLO para estabilidad del desempate numérico, no de las métricas.
        key = (m['costo_total'], m['FP'], -round(margen, 12), round(distancia, 12), umbrales)
        if mejor_key is None or key < mejor_key:
            mejor_key = key
            mejor = {'umbrales': dict(zip(VARIABLES, umbrales)),
                     'minimo_senales': 2, 'margen_minimo_entrenamiento': margen,
                     'metricas_entrenamiento': m}
    assert mejor is not None
    mejor.update(candidatos_evaluados=total, costo_fp=COSTO_FP, costo_fn=COSTO_FN,
                 rangos_entrenamiento={v: [min(xs), max(xs)] for v, xs in zip(VARIABLES, columnas)},
                 seleccion='Costo, FP, margen mínimo, centralidad, orden lexicográfico; solo casos 1–14.')
    return mejor


def predecir(relacion: Any, caballeria: Any, terreno: Any, modelo: dict) -> dict:
    """La función de inferencia NO recibe caso, conjunto ni resultado_real."""
    valores = validar_entrada(relacion, caballeria, terreno)
    senales = {v: x >= modelo['umbrales'][v] for v, x in zip(VARIABLES, valores)}
    cuenta = sum(senales.values())
    pred = int(cuenta >= modelo['minimo_senales'])
    fuera = [v for v, x in zip(VARIABLES, valores)
             if not modelo['rangos_entrenamiento'][v][0] <= x <= modelo['rangos_entrenamiento'][v][1]]
    return {'prediccion': pred, 'salida': 'SÍ' if pred else 'NO', 'senales': senales,
            'cantidad_senales': cuenta,
            'accion': 'Recomendar aceptar' if pred else 'Recomendar rehuir',
            'explicacion': f'{cuenta} de 3 condiciones favorables; se requieren {modelo["minimo_senales"]}.',
            'advertencias': (['Fuera del rango observado: ' + ', '.join(fuera) + '. Resultado no validado en esa zona.'] if fuera else [])}


def evaluar(filas: list[dict], modelo: dict) -> dict:
    reporte = {}
    for grupo in ('entrenamiento', 'prueba'):
        fs = [f for f in filas if f['conjunto'] == grupo]
        y = [f['resultado_real'] for f in fs]
        pred = [predecir(*(f[v] for v in VARIABLES), modelo)['prediccion'] for f in fs]
        permisivos = [int(any(f[v] >= modelo['umbrales'][v] for v in VARIABLES)) for f in fs]
        reporte[grupo] = {
            'sistema': metricas(y, pred),
            'siempre_no': metricas(y, [0] * len(fs)),
            'siempre_si': metricas(y, [1] * len(fs)),
            'variante_una_senal': metricas(y, permisivos),
            'errores': [f['caso'] for f, p in zip(fs, pred) if p != f['resultado_real']],
        }
    return reporte


def guardar_resultados(filas: list[dict], modelo: dict, reporte: dict, carpeta: Path) -> None:
    carpeta.mkdir(parents=True, exist_ok=True)
    (carpeta / 'metricas.json').write_text(json.dumps(reporte, ensure_ascii=False, indent=2), encoding='utf-8')
    campos = ['caso', 'conjunto', *VARIABLES, 'resultado_real', 'senales', 'prediccion', 'tipo']
    with (carpeta / 'predicciones_M01.csv').open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=campos); w.writeheader()
        for fila in filas:
            p = predecir(*(fila[v] for v in VARIABLES), modelo)
            tipo = {(1, 1): 'VP', (1, 0): 'FN', (0, 1): 'FP', (0, 0): 'VN'}[(fila['resultado_real'], p['prediccion'])]
            w.writerow({**fila, 'senales': p['cantidad_senales'], 'prediccion': p['prediccion'], 'tipo': tipo})


def resumen(reporte: dict) -> str:
    texto = []
    fmt = lambda x: 'No definida (denominador cero)' if x is None else f'{100 * x:.2f}%'
    for grupo, modelos in reporte.items():
        texto.append('\n' + grupo.upper())
        for nombre in ('sistema', 'siempre_no', 'siempre_si'):
            m = modelos[nombre]
            texto.append(f'{nombre}: VP={m["VP"]} FN={m["FN"]} FP={m["FP"]} VN={m["VN"]}; N={m["N"]}')
            texto.append('  ' + '; '.join(f'{k}={fmt(m[k])}' for k in ('exactitud', 'precision', 'sensibilidad', 'especificidad')))
            texto.append(f'  costo 5FP+FN={m["costo_total"]}; costo medio={m["costo_medio"]:.6f}')
        texto.append('Casos mal clasificados: ' + str(modelos['errores']))
    return '\n'.join(texto)


def ejecutar(ruta: str | Path, salida: str | Path) -> tuple[dict, dict]:
    filas = cargar_datos(ruta)
    entrenamiento = [f for f in filas if f['conjunto'] == 'entrenamiento']
    modelo = ajustar(entrenamiento)
    carpeta = Path(salida); carpeta.mkdir(parents=True, exist_ok=True)
    # Congelación antes de llamar a evaluar, que es quien usa las etiquetas de prueba.
    texto_modelo = json.dumps(modelo, ensure_ascii=False, indent=2)
    (carpeta / 'modelo_M01.json').write_text(texto_modelo, encoding='utf-8')
    (carpeta / 'modelo_M01.sha256').write_text(hashlib.sha256(texto_modelo.encode()).hexdigest() + '\n', encoding='utf-8')
    reporte = evaluar(filas, modelo)
    guardar_resultados(filas, modelo, reporte, carpeta)
    (carpeta / 'resumen.txt').write_text(resumen(reporte) + '\n', encoding='utf-8')
    return modelo, reporte



## 3. Selección y congelación de umbrales
Se prueban puntos medios calculados **solo con los 14 casos de entrenamiento**. Criterio: menor 5FP+FN; desempates por FP, margen, centralidad y orden de umbrales. Se guarda el modelo antes de evaluar prueba.

In [3]:
filas = cargar_datos(BASE / "datos" / "M01.csv")
entrenamiento = [f for f in filas if f["conjunto"] == "entrenamiento"]
modelo = ajustar(entrenamiento)
(BASE / "resultados").mkdir(exist_ok=True)
texto_modelo = json.dumps(modelo, ensure_ascii=False, indent=2)
(BASE / "resultados" / "modelo_M01.json").write_text(texto_modelo, encoding="utf-8")
print("Casos usados para ajustar:", [f["caso"] for f in entrenamiento])
print("Candidatos evaluados:", modelo["candidatos_evaluados"])
print("Umbrales congelados:", modelo["umbrales"])
print("Regla: SÍ con al menos dos de tres condiciones favorables.")
print("SHA-256:", hashlib.sha256(texto_modelo.encode()).hexdigest())

Casos usados para ajustar: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Candidatos evaluados: 1144
Umbrales congelados: {'relacion': 1.42, 'caballeria': 15.0, 'terreno': 6.5}
Regla: SÍ con al menos dos de tres condiciones favorables.
SHA-256: c672802dbc4707e3aedb4fc4b96a92857073f566be18417a928cd0deba44e396


## 4. Evaluación y comparación
Matrices: filas = realidad; columnas = predicción. Precisión sin predicciones positivas = **no definida**. No se cambian umbrales para mejorar los resultados de prueba.

In [4]:
reporte = evaluar(filas, modelo)
guardar_resultados(filas, modelo, reporte, BASE / "resultados")
print(resumen(reporte))
print("\nVariante de UNA señal, entrenamiento:")
print(reporte["entrenamiento"]["variante_una_senal"])
print("\nCasos: id | conjunto | real | predicción | señales | tipo")
for fila in filas:
    p = predecir(*(fila[v] for v in VARIABLES), modelo)
    tipo = {(1, 1): "VP", (1, 0): "FN", (0, 1): "FP", (0, 0): "VN"}[(fila["resultado_real"], p["prediccion"])]
    print(f'{fila["caso"]:2} | {fila["conjunto"]:13} | {fila["resultado_real"]} | {p["prediccion"]} | {p["cantidad_senales"]} | {tipo}')


ENTRENAMIENTO
sistema: VP=6 FN=0 FP=0 VN=8; N=14
  exactitud=100.00%; precision=100.00%; sensibilidad=100.00%; especificidad=100.00%
  costo 5FP+FN=0; costo medio=0.000000
siempre_no: VP=0 FN=6 FP=0 VN=8; N=14
  exactitud=57.14%; precision=No definida (denominador cero); sensibilidad=0.00%; especificidad=100.00%
  costo 5FP+FN=6; costo medio=0.428571
siempre_si: VP=6 FN=0 FP=8 VN=0; N=14
  exactitud=42.86%; precision=42.86%; sensibilidad=100.00%; especificidad=0.00%
  costo 5FP+FN=40; costo medio=2.857143
Casos mal clasificados: []

PRUEBA
sistema: VP=1 FN=2 FP=0 VN=3; N=6
  exactitud=66.67%; precision=100.00%; sensibilidad=33.33%; especificidad=100.00%
  costo 5FP+FN=2; costo medio=0.333333
siempre_no: VP=0 FN=3 FP=0 VN=3; N=6
  exactitud=50.00%; precision=No definida (denominador cero); sensibilidad=0.00%; especificidad=100.00%
  costo 5FP+FN=3; costo medio=0.500000
siempre_si: VP=3 FN=0 FP=3 VN=0; N=6
  exactitud=50.00%; precision=50.00%; sensibilidad=100.00%; especificidad=0.00%
 

## 5. Interpretación para la defensa
Entrenamiento: **14/14**; prueba: **4/6**, con 1 VP, 2 FN, 0 FP y 3 VN. Los casos 15 y 20 fallan porque solo cumplen caballería. El 100 % de precisión de prueba tiene denominador 1, no prueba fiabilidad general. La sensibilidad es 1/3. La base siempre NO acierta 3/6 y tiene tres FN. El costo final es 2 frente a 3 de esa base. No bajar umbrales viendo prueba: requeriría nueva validación y otra prueba independiente.

## 6. Caso nuevo · modificar estos tres valores
Puede usarse la celda como formulario de Colab o editar los números. No se solicita la etiqueta real. Entrada inválida → corregir, no clasificar como NO.

In [5]:
#@title Evaluar una batalla ofrecida
relacion = 1.60 #@param {type:"number"}
caballeria = 25 #@param {type:"number"}
terreno = 8 #@param {type:"number"}
try:
    salida = predecir(relacion, caballeria, terreno, modelo)
    print("PREDICCIÓN:", salida["salida"])
    print("ACCIÓN:", salida["accion"])
    for nombre, valor in zip(VARIABLES, (relacion, caballeria, terreno)):
        print(f'{nombre}: {valor} >= {modelo["umbrales"][nombre]} -> {salida["senales"][nombre]}')
    print(salida["explicacion"])
    for aviso in salida["advertencias"]: print("ADVERTENCIA:", aviso)
    print("Simulación didáctica; no es una garantía de victoria.")
except ValueError as error:
    print("ENTRADA INVÁLIDA:", error)

PREDICCIÓN: SÍ
ACCIÓN: Recomendar aceptar
relacion: 1.6 >= 1.42 -> True
caballeria: 25 >= 15.0 -> True
terreno: 8 >= 6.5 -> True
3 de 3 condiciones favorables; se requieren 2.
Simulación didáctica; no es una garantía de victoria.
